# Peek at MAG-based taxonomy ↔ biosamples

Worked-example queries linking metagenome-assembled genome (MAG) taxonomy and
quality results to biosamples via `nmdc_metadata.biosample_to_workflow_run`.

Covers:
- **GTDBTK bacterial classification** (`nmdc_results.gtdbtk_bacterial_summary`) — full lineage per MAG bin
- **CheckM quality statistics** (`nmdc_results.checkm_statistics`) — completeness, contamination

**Both directions:**
- §1–2: biosample → MAG classifications / quality for that sample
- §4–5: taxonomy substring / quality threshold → all biosamples

**Note:** GTDBTK archaeal (`nmdc_results.gtdbtk_archaeal_summary`) has no data in this corpus —
every file was a single-line placeholder. See §3.

In [1]:
spark = get_spark_session(app_name="peek_mag_taxonomy_links", tenant_name="nmdc")
print(f"Spark version: {spark.version}")

Spark version: 4.0.1


### Preflight: check available MAG taxonomy tables

In [2]:
existing = {r["tableName"] for r in spark.sql("SHOW TABLES IN nmdc_results").toPandas().to_dict("records")}

checks = {
    'gtdbtk_bacterial': 'gtdbtk_bacterial_summary',
    'checkm':           'checkm_statistics',
    'gtdbtk_archaeal':  'gtdbtk_archaeal_summary',
}

available = {}
for key, tbl in checks.items():
    found = tbl in existing
    available[key] = found
    status = "OK" if found else "MISSING"
    print(f"{status}: nmdc_results.{tbl}")

OK: nmdc_results.gtdbtk_bacterial_summary
OK: nmdc_results.checkm_statistics
MISSING: nmdc_results.gtdbtk_archaeal_summary


### Pick an anchor biosample

Find a biosample that has at least one GTDBTK bacterial result via the workflow bridge.

In [3]:
_anchor_df = spark.sql("""
    SELECT b2wr.biosample_id, b2wr.workflow_run_id, b2wr.workflow_type
    FROM   nmdc_results.gtdbtk_bacterial_summary g
    JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
             ON b2wr.workflow_run_id = g.workflow_run_id
    LIMIT 1
""").toPandas()
if _anchor_df.empty:
    raise RuntimeError("no gtdbtk_bacterial_summary row joins to biosample_to_workflow_run — verify both tables are populated")
anchor = _anchor_df.iloc[0]
BIOSAMPLE_ID  = anchor["biosample_id"]
GTDBTK_RUN_ID = anchor["workflow_run_id"]
print(anchor.to_string())

biosample_id           nmdc:bsm-11-zh14q280
workflow_run_id    nmdc:wfmag-11-93d4y473.1
workflow_type             nmdc:MagsAnalysis


### 1. Biosample → GTDBTK bacterial MAG classifications

Each row is one MAG (bin) recovered from this biosample's metagenome assembly.
The `classification` column is a full GTDBTK lineage string.

In [4]:
spark.sql(f"""
    SELECT g.name AS bin_id, g.classification, g.fastani_reference,
           ROUND(g.fastani_ani, 2) AS fastani_ani, ROUND(g.fastani_af, 2) AS fastani_af
    FROM   nmdc_results.gtdbtk_bacterial_summary g
    WHERE  g.workflow_run_id = '{GTDBTK_RUN_ID}'
    ORDER  BY g.fastani_ani DESC NULLS LAST
""").toPandas()

,bin_id,classification,fastani_reference,fastani_ani,fastani_af
0,bins.4,d__Bacteria;p__Acidobacteriota;c__Acidobacteri...,NaN,NaN,NaN


### 2. Biosample → CheckM MAG quality

Completeness ≥ 90% and contamination ≤ 5% is the standard 'high-quality draft' threshold (MIMAG).

In [5]:
if not available['checkm']:
    print("SKIP: checkm_statistics not available")
else:
    checkm_run = spark.sql(f"""
        SELECT b2wr.workflow_run_id
        FROM   nmdc_metadata.biosample_to_workflow_run b2wr
        JOIN   nmdc_results.checkm_statistics c ON c.workflow_run_id = b2wr.workflow_run_id
        WHERE  b2wr.biosample_id = '{BIOSAMPLE_ID}'
        LIMIT 1
    """).toPandas()
    if checkm_run.empty:
        print(f"No CheckM results for {BIOSAMPLE_ID}")
    else:
        CHECKM_RUN_ID = checkm_run.iloc[0]["workflow_run_id"]
        display(spark.sql(f"""
            SELECT bin_id, marker_lineage,
                   ROUND(completeness, 1)        AS completeness,
                   ROUND(contamination, 1)       AS contamination,
                   ROUND(strain_heterogeneity, 1) AS strain_heterogeneity,
                   n_markers
            FROM   nmdc_results.checkm_statistics
            WHERE  workflow_run_id = '{CHECKM_RUN_ID}'
            ORDER  BY completeness DESC, contamination ASC
        """).toPandas())

### 3. GTDBTK archaeal — not available

Every GTDBTK Archaeal file in the NMDC corpus is a single-line placeholder:
```
No Archaeal Results for nmdc:wfmag-...
```
This means none of the 3,535 MAG workflow runs recovered any archaeal bins. Either the
upstream binning pipeline filters out archaeal bins before GTDB-Tk, or none of the
assemblies in this load cycle recovered archaea. The table
`nmdc_results.gtdbtk_archaeal_summary` was not ingested.

When archaeal data becomes available, the query pattern is identical to §1 but
referencing `gtdbtk_archaeal_summary`.

### 4. Taxonomy string → biosamples (GTDBTK)

Find all biosamples that recovered at least one MAG classified to a given taxonomic group.
Filter on a substring of the GTDBTK classification lineage string.

In [6]:
TARGET_PHYLUM = "Bacteroidota"  # substitute any taxon substring from the GTDBTK lineage

spark.sql(f"""
    SELECT b2wr.biosample_id,
           bsm.env_broad_scale_term_id,
           bsm.geo_loc_name_has_raw_value,
           COUNT(*) AS n_matching_mags,
           COLLECT_LIST(g.name) AS bin_ids
    FROM   nmdc_results.gtdbtk_bacterial_summary g
    JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
             ON b2wr.workflow_run_id = g.workflow_run_id
    JOIN   nmdc_metadata.biosample_set bsm
             ON bsm.id = b2wr.biosample_id
    WHERE  g.classification LIKE '%{TARGET_PHYLUM}%'
    GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id, bsm.geo_loc_name_has_raw_value
    ORDER BY n_matching_mags DESC
    LIMIT 20
""").toPandas()

,biosample_id,env_broad_scale_term_id,geo_loc_name_has_raw_value,n_matching_mags,bin_ids
0,nmdc:bsm-11-yfkh8t53,ENVO:01000252,USA: Wisconsin,22,"[bins.101, bins.103, bins.107, bins.111, bins...."
1,nmdc:bsm-11-h87jr161,ENVO:01000252,USA: Wisconsin,20,"[bins.100, bins.103, bins.109, bins.11, bins.1..."
2,nmdc:bsm-11-0ynadq83,ENVO:01000252,USA: Wisconsin,19,"[bins.108, bins.113, bins.116, bins.16, bins.3..."
3,nmdc:bsm-11-6r9zc028,ENVO:01000252,USA: Wisconsin,19,"[bins.101, bins.102, bins.107, bins.114, bins...."
4,nmdc:bsm-11-dzggr533,ENVO:01000252,USA: Wisconsin,19,"[bins.118, bins.128, bins.133, bins.141, bins...."
5,nmdc:bsm-11-hbw7d587,ENVO:01000252,USA: Wisconsin,19,"[bins.110, bins.136, bins.137, bins.14, bins.1..."
6,nmdc:bsm-11-4dhxs983,ENVO:01000252,USA: Wisconsin,18,"[bins.108, bins.115, bins.119, bins.124, bins...."
7,nmdc:bsm-11-96r7vz43,ENVO:01000252,USA: Wisconsin,18,"[bins.100, bins.110, bins.131, bins.139, bins...."
8,nmdc:bsm-11-bwkb2749,ENVO:01000252,USA: Wisconsin,18,"[bins.1, bins.114, bins.117, bins.119, bins.13..."
9,nmdc:bsm-11-vfpcpt51,ENVO:01000252,USA: Wisconsin,18,"[bins.113, bins.118, bins.133, bins.144, bins...."


### 5. CheckM quality threshold → biosamples

Find biosamples that recovered high-quality MAGs (completeness ≥ 90, contamination ≤ 5 —
MIMAG standard). Returns the biosample and the count of high-quality bins.

In [7]:
spark.sql("""
    SELECT b2wr.biosample_id,
           bsm.env_broad_scale_term_id,
           bsm.geo_loc_name_has_raw_value,
           COUNT(*) AS n_hq_mags,
           ROUND(AVG(c.completeness), 1)  AS avg_completeness,
           ROUND(AVG(c.contamination), 2) AS avg_contamination
    FROM   nmdc_results.checkm_statistics c
    JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
             ON b2wr.workflow_run_id = c.workflow_run_id
    JOIN   nmdc_metadata.biosample_set bsm
             ON bsm.id = b2wr.biosample_id
    WHERE  c.completeness >= 90
      AND  c.contamination <= 5
    GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id, bsm.geo_loc_name_has_raw_value
    ORDER BY n_hq_mags DESC
    LIMIT 20
""").toPandas()

,biosample_id,env_broad_scale_term_id,geo_loc_name_has_raw_value,n_hq_mags,avg_completeness,avg_contamination
0,nmdc:bsm-13-mn6sjr37,ENVO:01000174,"USA: Minnesota, Marcel Experimental Forest, Sp...",25,94.9,2.14
1,nmdc:bsm-13-7q44hn24,ENVO:01000174,"USA: Minnesota, Marcel Experimental Forest, Sp...",24,95.7,1.98
2,nmdc:bsm-13-t4gewc31,ENVO:01000174,"USA: Minnesota, Marcel Experimental Forest, Sp...",23,94.3,1.93
3,nmdc:bsm-13-4g22tp63,ENVO:01000174,"USA: Minnesota, Marcel Experimental Forest, Sp...",21,95.1,2.08
4,nmdc:bsm-13-z9ejh210,ENVO:01000174,"USA: Minnesota, Marcel Experimental Forest, Sp...",20,95.6,2.52
5,nmdc:bsm-11-afygsz12,ENVO:01001002,"USA: California, Davis",19,94.2,0.88
6,nmdc:bsm-13-ckbyrj91,ENVO:01000174,"USA: Minnesota, Marcel Experimental Forest, Sp...",19,96.2,2.70
7,nmdc:bsm-12-zyae7e18,ENVO:00000446,"USA: Michigan, Houghton",19,95.4,1.94
8,nmdc:bsm-13-p0q6k325,ENVO:01000174,"USA: Minnesota, Marcel Experimental Forest, Sp...",19,95.7,1.96
9,nmdc:bsm-13-h067tp29,ENVO:01000174,"USA: Minnesota, Marcel Experimental Forest, Sp...",18,96.3,2.44
